In [1]:
import pandas as pd
import numpy as np


## 1. Generate Synthetic data

In [2]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)

# 1. Shrink domain to [-1, 1] to make it easier to learn
X = rng.uniform(-1, 1, size=(10000, 2))
# 2. Reduce noise slightly
noise = rng.normal(0, 0.05, size=10000)

# 3. Simple "Bowl" shape: y = x1^2 + x2^2 + noise
y = (
    X[:, 0]**2 + 
    X[:, 1]**2 + 
    noise
).reshape(-1, 1)

print(f"X Shape : {X.shape}")
print(f"y Shape : {y.shape}")

data = pd.DataFrame(np.hstack((X, y)), columns=["x1", "x2", "y"])
data.head()

X Shape : (10000, 2)
y Shape : (10000, 1)


,x1,x2,y
0,0.547912,-0.122243,0.223862
1,0.717196,0.394736,0.650452
2,-0.811645,0.951245,1.476215
3,0.522279,0.572129,0.702184
4,-0.743773,-0.099228,0.573669


In [3]:
import plotly.express as px

fig = px.scatter_3d(data, x="x1", y="x2",z="y", color="y", 
                    title="Synthetic Regression Data",
                    labels={"x1": "Feature 1", "x2": "Feature 2", "y": "Target"})
fig.show()

## 2. weight Initialization

In [4]:
def initialize_parameters(dims, seed = 42):
    
    rng = np.random.default_rng(seed)
    parameters = {}
    
    # Xavier normal initialization to keep weights at a reasonable scale. 
    for l in range(1, len(dims)):
        
        fan_in = dims[l-1]
        fan_out = dims[l]
        
        weight_scale = np.sqrt(2 / (fan_in + fan_out))
        
        parameters[f"W{l}"] = rng.normal(0 , weight_scale, (fan_in, fan_out))
        parameters[f"b{l}"] = np.zeros((1, fan_out))
    
    return parameters



In [8]:
ini_parameters = initialize_parameters([2, 2, 1])
ini_parameters['W2'].shape


(2, 1)

## 3. Linear Forward

In [12]:
def linear_forward(A_prev, W, b):
    """
    A_prev: activations from the previous layer.
    W: weight matrix for this layer.
    b: bias vector for this layer."""
    
    Z = np.dot(A_prev, W) + b
    
    return Z

## 4. Forward Propagation

In [14]:
def activation(Z):       return Z
def activation_grad(Z):  return np.ones_like(Z)


In [ ]:
def forward_propagation(X, params):
    A = np.atleast_2d(X)                  # (m, 2)
    cache = {"A0": A}
    L = len(params) // 2
    for l in range(1, L + 1):
        Z = A @ params[f"W{l}"] + params[f"b{l}"]
        A = Z if l == L else activation(Z)
        cache[f"Z{l}"], cache[f"A{l}"] = Z, A   # A1 lives here, shape (m, 2)
    return A, cache

In [16]:
def forward_propagation(X, params, show=False):
    """
    X: input data.
    parameters: dictionary containing the weights and biases of all layers."""
    
    A = np.atleast_2d(X)
    cache = {"A0" : A}
    layer_count = len(params) // 2 # Number of layers
    
    for l in range(1, layer_count +1 ):
        A_prev = A
        W = params[f"W{l}"]
        b = params[f"b{l}"]
        
        if show:
            print(f"A{l-1} : {A_prev}")
            print(f"W{l} : {W}")
            print(f"b{l} : {b}")
            print(f"A{l}: {A}")
            print("----"*20)
        
        Z = linear_forward(A_prev, W, b)

        if l == layer_count:
            A = Z # Output layer: Linear for regression 
        else:
            A = activation(Z)
            
        cache[f"Z{l}"], cache[f"A{l}"] = Z, A
    
    return A, cache

In [17]:
y_hat, c = forward_propagation(X[0], ini_parameters)
print("Y predict : ",y_hat," | ",y_hat.shape)
print(c)

Y predict :  [[0.43010921]]  |  (1, 1)
{'A0': array([[ 0.5479121 , -0.12224312]]), 'Z1': array([[ 0.05318905, -0.48422491]]), 'A1': array([[ 0.05318905, -0.48422491]]), 'Z2': array([[0.43010921]]), 'A2': array([[0.43010921]])}
